In [ ]:
import os
import time
import copy
import csv
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler

# --- Configuration ---
DATA_DIR = Path("../data/labeled_dataset")
BASE_OUTPUT_DIR = Path("../runs/comparison")
NUM_CLASSES = 8
BATCH_SIZE = 16 
NUM_EPOCHS = 5 # As requested for rapid comparison
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_NAMES = ['a_bikes', 'b_moto', 'c_pass', 'd_light_comm', 
               'e_heavy_rigid', 'f_articulated', 'g_bus', 'h_agri']

# --- 1. Data Setup (Fixed for all models) ---
def get_dataloaders():
    # Standard ImageNet normalization works for all these pretrained models
    norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    
    transforms_dict = {
        'train': transforms.Compose([
            transforms.Resize((236, 236)),
            transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
            transforms.ToTensor(),
            norm
        ]),
        'val': transforms.Compose([
            transforms.Resize((236, 236)),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            norm
        ])
    }

    full_dataset = datasets.ImageFolder(DATA_DIR, transform=transforms_dict['train'])
    
    # Fix the seed for reproducible splits
    generator = torch.Generator().manual_seed(42)
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size], generator=generator)
    
    val_dataset.dataset.transform = transforms_dict['val']
    
    # Weighted Sampler
    train_targets = [full_dataset.targets[i] for i in train_dataset.indices]
    class_counts = np.bincount(train_targets)
    class_weights = 1. / class_counts
    sample_weights = [class_weights[t] for t in train_targets]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

    return {
        'train': DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4),
        'val': DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    }

# --- 2. Model Factory ---
def get_model(model_name):
    print(f"Initializing {model_name}...")
    
    if model_name == "resnet50":
        model = models.resnet50(weights='DEFAULT')
        # ResNet: Linear layer is 'fc'
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
        
    elif model_name == "efficientnet_v2_s":
        model = models.efficientnet_v2_s(weights='DEFAULT')
        # EfficientNet: Classifier is a sequential block, last layer is [1]
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
        
    elif model_name == "densenet121":
        model = models.densenet121(weights='DEFAULT')
        # DenseNet: Linear layer is 'classifier'
        model.classifier = nn.Linear(model.classifier.in_features, NUM_CLASSES)
        
    elif model_name == "vit_b_16":
        model = models.vit_b_16(weights='DEFAULT')
        # ViT: Heads is a sequential block
        model.heads.head = nn.Linear(model.heads.head.in_features, NUM_CLASSES)
        
    elif model_name == "convnext_base": # Your champion
        model = models.convnext_base(weights='DEFAULT')
        model.classifier[2] = nn.Linear(model.classifier[2].in_features, NUM_CLASSES)

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return model.to(DEVICE)

# --- 3. Generic Training Function ---
def train_session(model_name, dataloaders):
    save_dir = BASE_OUTPUT_DIR / model_name
    save_dir.mkdir(parents=True, exist_ok=True)
    
    model = get_model(model_name)
    
    # Standardize loss and optimizer for fair comparison
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_f1 = 0.0
    history = []

    print(f"\n--- Starting Training: {model_name} ---")

    for epoch in range(NUM_EPOCHS):
        print(f'Epoch {epoch+1}/{NUM_EPOCHS}')
        
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            all_preds = []
            all_labels = []

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_f1 = f1_score(all_labels, all_preds, average='macro')
            epoch_acc = np.mean(np.array(all_preds) == np.array(all_labels))
            
            # Print only Val stats to keep console clean
            if phase == 'val':
                print(f'  Val Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} Macro-F1: {epoch_f1:.4f}')

            history.append({
                'model': model_name,
                'epoch': epoch + 1,
                'phase': phase,
                'loss': epoch_loss,
                'accuracy': epoch_acc,
                'f1_macro': epoch_f1
            })

            if phase == 'val' and epoch_f1 > best_f1:
                best_f1 = epoch_f1
                best_model_wts = copy.deepcopy(model.state_dict())
                torch.save(model.state_dict(), save_dir / "best_model.pth")

    time_elapsed = time.time() - since
    print(f'Completed {model_name} in {time_elapsed:.0f}s. Best F1: {best_f1:.4f}')

    # --- Save Artifacts ---
    model.load_state_dict(best_model_wts)
    pd.DataFrame(history).to_csv(save_dir / "training_log.csv", index=False)
    
    # Confusion Matrix
    model.eval()
    y_true, y_pred = [], []
    with torch.inference_mode():
        for inputs, labels in dataloaders['val']:
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{model_name} Confusion Matrix (F1: {best_f1:.2f})')
    plt.savefig(save_dir / "confusion_matrix.png")
    plt.close()

    report = classification_report(y_true, y_pred, target_names=CLASS_NAMES)
    with open(save_dir / "classification_report.txt", "w") as f:
        f.write(report)

# --- 4. Main Execution Loop ---
if __name__ == "__main__":
    # Models to compare
    models_to_train = [
        "resnet50",         # The classic baseline
        "efficientnet_v2_s",# The efficiency baseline
        "densenet121",      # Feature reuse baseline
        "vit_b_16",         # The Transformer baseline (Patch-based)
        "convnext_base"     # Your Proposed Solution (CNN + Transformer insights)
    ]
    
    print("Loading Data (Shared Split)...")
    loaders = get_dataloaders()
    
    for model_name in models_to_train:
        try:
            train_session(model_name, loaders)
        except Exception as e:
            print(f"Failed to train {model_name}: {e}")
            
    print("\nAll comparisons finished. Results in runs/comparison/")

Loading Data (Shared Split)...
Initializing resnet50...


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\Karolina/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:43<00:00, 2.37MB/s]



--- Starting Training: resnet50 ---
Epoch 1/5
  Val Loss: 0.6389 Acc: 0.9489 Macro-F1: 0.8394
Epoch 2/5
  Val Loss: 0.5975 Acc: 0.9467 Macro-F1: 0.8253
Epoch 3/5
  Val Loss: 0.5923 Acc: 0.9511 Macro-F1: 0.8337
Epoch 4/5
  Val Loss: 0.5932 Acc: 0.9533 Macro-F1: 0.8408
Epoch 5/5
  Val Loss: 0.5886 Acc: 0.9511 Macro-F1: 0.8351
Completed resnet50 in 1523s. Best F1: 0.8408
Initializing efficientnet_v2_s...


Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to C:\Users\Karolina/.cache\torch\hub\checkpoints\efficientnet_v2_s-dd5fe13b.pth
100%|██████████| 82.7M/82.7M [00:42<00:00, 2.06MB/s]



--- Starting Training: efficientnet_v2_s ---
Epoch 1/5
  Val Loss: 0.6181 Acc: 0.9422 Macro-F1: 0.8499
Epoch 2/5
  Val Loss: 0.5891 Acc: 0.9533 Macro-F1: 0.8643
Epoch 3/5
  Val Loss: 0.5536 Acc: 0.9689 Macro-F1: 0.8861
Epoch 4/5
  Val Loss: 0.5596 Acc: 0.9600 Macro-F1: 0.8599
Epoch 5/5
  Val Loss: 0.5521 Acc: 0.9644 Macro-F1: 0.8701
Completed efficientnet_v2_s in 1812s. Best F1: 0.8861
Initializing densenet121...


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to C:\Users\Karolina/.cache\torch\hub\checkpoints\densenet121-a639ec97.pth
100%|██████████| 30.8M/30.8M [00:09<00:00, 3.58MB/s]



--- Starting Training: densenet121 ---
Epoch 1/5
  Val Loss: 0.6868 Acc: 0.9111 Macro-F1: 0.8123
Epoch 2/5
  Val Loss: 0.5859 Acc: 0.9578 Macro-F1: 0.8356
Epoch 3/5
  Val Loss: 0.5705 Acc: 0.9644 Macro-F1: 0.8559
Epoch 4/5
  Val Loss: 0.5672 Acc: 0.9644 Macro-F1: 0.8502
Epoch 5/5
  Val Loss: 0.5743 Acc: 0.9600 Macro-F1: 0.8416
Completed densenet121 in 1511s. Best F1: 0.8559
Initializing vit_b_16...


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to C:\Users\Karolina/.cache\torch\hub\checkpoints\vit_b_16-c867db91.pth
100%|██████████| 330M/330M [02:14<00:00, 2.57MB/s] 



--- Starting Training: vit_b_16 ---
Epoch 1/5
  Val Loss: 0.6714 Acc: 0.9022 Macro-F1: 0.8054
Epoch 2/5
  Val Loss: 0.6108 Acc: 0.9378 Macro-F1: 0.7923
Epoch 3/5
  Val Loss: 0.5453 Acc: 0.9667 Macro-F1: 0.8596
Epoch 4/5
  Val Loss: 0.5438 Acc: 0.9644 Macro-F1: 0.8550
Epoch 5/5


KeyboardInterrupt: 